## 🤝 04 — Multi-Agent Orchestration (Workflows)

**Goal**: Build workflows that orchestrate multiple agents


In [9]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath("..") if "notebooks" in os.getcwd() else ".")

from utils.config import get_foundry_client, get_env
from agent_framework import Agent, WorkflowBuilder, AgentResponse
from agent_framework.orchestrations import SequentialBuilder
from agent_framework.orchestrations import ConcurrentBuilder
from typing import cast

client = get_foundry_client()
print("✅ Connected to Foundry")

✅ Connected to Foundry


---
### Part A — Sequential Workflow (Writer → Reviewer) ✍️
A simple two-step pipeline: one agent writes, another reviews.

In [6]:
# ── A1. Define agents ────────────────────────────────────────────────────────
from agent_framework import Message

writer = client.as_agent(
    name="Writer",
    instructions=(
        "You are a concise copywriter. Write a single, punchy marketing "
        "paragraph based on the user's prompt. Be creative and engaging."
    ),
)

reviewer = client.as_agent(
    name="Reviewer",
    instructions=(
        "You are a content reviewer. Review the previous assistant message. "
        "Provide brief, actionable feedback: what's good and what to improve. "
        "Then provide a final polished version taking into consideration the feedback provided."
    ),
)

print("✍️ Writer -> 📝 Reviewer — agents created")

# 2) Build sequential workflow: writer -> reviewer
workflow = SequentialBuilder(participants=[writer, reviewer]).build()

# 3) Run and collect outputs
outputs: list[list[Message]] = []
async for event in workflow.run("Write a tagline for a budget-friendly eBike.", stream=True):
    if event.type == "output":
        outputs.append(cast(list[Message], event.data))

if outputs:
    print("===== Final Conversation =====")
    for i, msg in enumerate(outputs[-1], start=1):
        name = msg.author_name or ("assistant" if msg.role == "assistant" else "user")
        print(f"{'-' * 60}\n{i:02d} [{name}]\n{msg.text}")

✍️ Writer -> 📝 Reviewer — agents created
===== Final Conversation =====
------------------------------------------------------------
01 [user]
Write a tagline for a budget-friendly eBike.
------------------------------------------------------------
02 [Writer]
"Zoom into Savings: Eco-Friendly Rides Without the Price Tag!" 🚴‍♀️⚡
------------------------------------------------------------
03 [Reviewer]
**Feedback:**

- **What's Good:** The tagline is engaging and highlights both the affordability and eco-friendliness of the eBike. The use of the word "Zoom" adds a sense of excitement and speed.
  
- **What to Improve:** The tagline could be slightly more concise and sharper by removing unnecessary words. The emoji use might not be suitable for all professional contexts.

**Polished Version:**

"Zoom into Savings: Eco-Friendly Rides on a Budget!"


---
### Part B — Graph-Based Workflow (WorkflowBuilder - Researcher, Analyst and Final Reviewer) 📊
Build a more complex workflow with explicit edges and conditional routing.

In [7]:
# ── B1. Define specialized agents ────────────────────────────────────────────
researcher = client.as_agent(
    name="Researcher",
    instructions=(
        "You are a researcher. Gather key facts and data points about the "
        "given topic. Present findings as a structured brief."
    ),
)

analyst = client.as_agent(
    name="Analyst",
    instructions=(
        "You are a business analyst. Take the research brief and produce "
        "actionable insights with recommendations. Be specific and concise."
    ),
)

final_reviewer = client.as_agent(
    name="FinalReviewer",
    instructions=(
        "You are a senior reviewer. Take the analysis and produce a polished "
        "executive summary suitable for C-level presentation. Use bullet points."
    ),
)

print("🔬 Researcher, 📊 Analyst, 📝 FinalReviewer — agents created")

🔬 Researcher, 📊 Analyst, 📝 FinalReviewer — agents created


In [10]:
# ── B2. Build graph-based workflow ───────────────────────────────────────────
graph_workflow = (
    WorkflowBuilder(start_executor=researcher)
    .add_edge(researcher, analyst)
    .add_edge(analyst, final_reviewer)
    .build()
)

print("📊 Graph workflow: Researcher → Analyst → FinalReviewer")

events = await graph_workflow.run(
    "Analyze the competitive landscape for enterprise AI agent platforms in 2026."
)

outputs = events.get_outputs()
# The outputs of the workflow are whatever the agents produce. So the outputs are expected to be a list
# of `AgentResponse` from the agents in the workflow.
outputs = cast(list[AgentResponse], outputs)
for output in outputs:
    print(f"{output.messages[0].author_name}: {output.text}\n")

📊 Graph workflow: Researcher → Analyst → FinalReviewer
Researcher: ### Competitive Landscape for Enterprise AI Agent Platforms in 2026

#### Market Overview
- **Growth Projections**: The enterprise AI agent platform market is expected to see robust growth due to increased adoption across varying industries driven by digital transformation and the need for automated solutions.
- **Key Drivers**: Optimization of business processes, advancement in natural language processing (NLP), increased investment in AI technologies, and demand for enhanced customer interaction are pivotal growth drivers.

#### Major Players
1. **Microsoft**
   - **Platform**: Azure AI
   - **Strengths**: Robust cloud infrastructure, integration with existing enterprise tools, strong developer community.
   - **Strategic Moves**: Continued investments in AI research and development, emphasis on security and compliance.

2. **Google**
   - **Platform**: Google Cloud AI
   - **Strengths**: Superior machine learning alg

---
### Part C — Concurrent Workflow (ConcurrentBuilder - Researcher, Marketer, Legal) 📊
Build a more Concurrent fan-out/fan-in with default aggregator.

In [11]:
from typing import Any


researcher = Agent(
    client=client,
    instructions=(
        "You're an expert market and product researcher. Given a prompt, provide concise, factual insights,"
        " opportunities, and risks."
    ),
    name="researcher",
)

marketer = Agent(
    client=client,
    instructions=(
        "You're a creative marketing strategist. Craft compelling value propositions and target messaging"
        " aligned to the prompt."
    ),
    name="marketer",
)

legal = Agent(
    client=client,
    instructions=(
        "You're a cautious legal/compliance reviewer. Highlight constraints, disclaimers, and policy concerns"
        " based on the prompt."
    ),
    name="legal",
)

# 2) Build a concurrent workflow
# Participants are either Agents (type of SupportsAgentRun) or Executors
workflow = ConcurrentBuilder(participants=[researcher, marketer, legal]).build()

# 3) Run with a single prompt and pretty-print the final combined messages
events = await workflow.run("We are launching a new budget-friendly electric bike for urban commuters.")
outputs = events.get_outputs()

if outputs:
    print("===== Final Aggregated Conversation (messages) =====")
    for output in outputs:
        messages: list[Message] | Any = output
        for i, msg in enumerate(messages, start=1):
            name = msg.author_name if msg.author_name else "user"
            print(f"{'-' * 60}\n\n{i:02d} [{name}]:\n{msg.text}")


===== Final Aggregated Conversation (messages) =====
------------------------------------------------------------

01 [user]:
We are launching a new budget-friendly electric bike for urban commuters.
------------------------------------------------------------

02 [researcher]:
**Insights:**

1. **Market Growth:**
   - The global e-bike market is projected to grow significantly, driven by urbanization and environmental concerns.
   - Budget-friendly options are gaining popularity as consumers seek cost-effective and sustainable commuting solutions.

2. **Target Audience:**
   - Urban dwellers looking for economical and eco-friendly commuting options.
   - Young professionals, students, and eco-conscious individuals are key demographics.

3. **Competitor Landscape:**
   - Established brands like Rad Power Bikes and Ancheer dominate budget segments.
   - New entrants can differentiate through innovative features, design, and pricing strategies.

**Opportunities:**

1. **Design Customizat

---
### ✅ What you demonstrated
| Capability | Agent Framework Feature |
|---|---|
| Sequential pipelines | `SequentialBuilder(participants=[...]).build()` |
| Graph-based orchestration | `WorkflowBuilder().add_edge(a, b).build()` |
| Concurrent orchestration | `ConcurrentBuilder(participants=[...]).build()` |

**Next →** [05_evaluate_and_deploy.ipynb](./05_evaluate_and_deploy.ipynb) — evaluate quality and deploy as a hosted agent.